
## Notebook 3 — SQL Analysis & EDA


### Setup

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "bangalore_traffic", "user": "postgres", "password": "YOUR_PASSWORD_HERE",
}
DB_URL = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
          f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
engine = create_engine(DB_URL)

def sql(query, **kwargs):
    return pd.read_sql(text(query), engine, params=kwargs)

print(" Ready.")


 Ready.


---
##  Section 1 — Congestion & Traffic Patterns


### Top 10 Worst Junctions (Morning Rush 8–10 AM)

In [3]:
query = """
SELECT
    junction_label,
    ROUND(AVG(congestion_index)::numeric, 3)   AS avg_congestion,
    ROUND(AVG(current_speed)::numeric, 1)       AS avg_speed_kmh,
    ROUND(AVG(free_flow_speed)::numeric, 1)     AS free_flow_kmh,
    COUNT(*)                                    AS observations
FROM traffic_speeds_clean
WHERE hour BETWEEN 8 AND 10
GROUP BY junction_label
ORDER BY avg_congestion DESC
LIMIT 10;
"""
worst_am = sql(query)
worst_am


,junction_label,avg_congestion,avg_speed_kmh,free_flow_kmh,observations
0,MG Road,0.664,10.1,30.0,90
1,Koramangala,0.663,9.4,28.0,90
2,Outer Ring Road KR Puram,0.663,11.5,34.0,90
3,Hebbal,0.662,11.8,35.0,90
4,HSR Layout,0.661,9.2,27.0,90
5,Mahadevapura,0.661,10.9,32.0,90
6,Whitefield,0.659,12.6,37.0,90
7,BTM Layout,0.658,8.6,25.0,90
8,Marathahalli,0.657,10.6,31.0,90
9,Silk Board,0.657,8.9,26.0,90


### Morning vs Evening Peak — Side-by-Side

In [4]:
query = """
SELECT
    junction_label,
    ROUND(AVG(CASE WHEN hour BETWEEN 8 AND 10 THEN congestion_index END)::numeric, 3) AS morning_congestion,
    ROUND(AVG(CASE WHEN hour BETWEEN 18 AND 20 THEN congestion_index END)::numeric, 3) AS evening_congestion
FROM traffic_speeds_clean
GROUP BY junction_label
ORDER BY GREATEST(
    AVG(CASE WHEN hour BETWEEN 8 AND 10 THEN congestion_index END),
    AVG(CASE WHEN hour BETWEEN 18 AND 20 THEN congestion_index END)
) DESC;
"""
peak_compare = sql(query)
peak_compare


,junction_label,morning_congestion,evening_congestion
0,MG Road,0.664,0.655
1,Koramangala,0.663,0.655
2,Outer Ring Road KR Puram,0.663,0.660
3,Hebbal,0.662,0.652
4,HSR Layout,0.661,0.645
5,Mahadevapura,0.661,0.653
6,Whitefield,0.659,0.656
7,BTM Layout,0.658,0.643
8,Silk Board,0.657,0.651
9,Marathahalli,0.657,0.649


### Weekday vs Weekend Traffic by Area

In [5]:
query = """
SELECT
    junction_label,
    ROUND(AVG(CASE WHEN NOT is_weekend THEN congestion_index END)::numeric, 3) AS weekday_congestion,
    ROUND(AVG(CASE WHEN is_weekend THEN congestion_index END)::numeric, 3)     AS weekend_congestion
FROM traffic_speeds_clean
GROUP BY junction_label
ORDER BY weekday_congestion DESC;
"""
wd_we = sql(query)
wd_we


,junction_label,weekday_congestion,weekend_congestion
0,Outer Ring Road KR Puram,0.418,0.292
1,MG Road,0.418,0.286
2,HSR Layout,0.417,0.278
3,Silk Board,0.416,0.285
4,Hebbal,0.415,0.288
5,Mahadevapura,0.415,0.290
6,BTM Layout,0.415,0.281
7,Koramangala,0.415,0.288
8,Marathahalli,0.415,0.286
9,Whitefield,0.414,0.288


### Optimal Departure Time per Junction

In [6]:
query = """
SELECT DISTINCT ON (junction_label)
    junction_label,
    hour,
    ROUND(AVG(congestion_index)::numeric, 3) AS avg_congestion
FROM traffic_speeds_clean
GROUP BY junction_label, hour
ORDER BY junction_label, avg_congestion ASC;
"""
best_time = sql(query)
best_time


,junction_label,hour,avg_congestion
0,BTM Layout,1,0.087
1,HSR Layout,1,0.078
2,Hebbal,3,0.087
3,Koramangala,2,0.079
4,MG Road,3,0.087
5,Mahadevapura,3,0.083
6,Marathahalli,3,0.087
7,Outer Ring Road KR Puram,3,0.091
8,Silk Board,3,0.088
9,Whitefield,2,0.094


---
##  Section 2 — Weather Impact on Traffic


###  Average Congestion by Rain Category

In [7]:
query = """
SELECT
    w.rain_category,
    ROUND(AVG(t.congestion_index)::numeric, 3) AS avg_congestion,
    ROUND(AVG(t.current_speed)::numeric, 1)    AS avg_speed_kmh,
    COUNT(*)                                    AS data_points
FROM traffic_speeds_clean t
JOIN weather_clean w
  ON t.fetched_at::date = w.date
 AND EXTRACT(HOUR FROM t.fetched_at) = w.hour
GROUP BY w.rain_category
ORDER BY avg_congestion DESC;
"""
rain_impact = sql(query)
rain_impact


,rain_category,avg_congestion,avg_speed_kmh,data_points


### 2.2 Rainfall Tipping Point — Where Does Congestion Spike?

In [8]:
query = """
SELECT
    FLOOR(w.precipitation_mm / 2) * 2          AS rain_bucket_mm,
    ROUND(AVG(t.congestion_index)::numeric, 3) AS avg_congestion,
    COUNT(*)                                    AS obs
FROM traffic_speeds_clean t
JOIN weather_clean w
  ON t.fetched_at::date = w.date
 AND EXTRACT(HOUR FROM t.fetched_at) = w.hour
WHERE w.precipitation_mm <= 40
GROUP BY rain_bucket_mm
ORDER BY rain_bucket_mm;
"""
tipping = sql(query)
tipping


,rain_bucket_mm,avg_congestion,obs


###  Monsoon Time Cost — Extra Hours Lost Per Week

In [9]:
query = """
SELECT
    w.is_monsoon,
    ROUND(AVG(t.current_travel_time)::numeric, 1)   AS avg_travel_sec,
    ROUND(AVG(t.free_flow_travel_time)::numeric, 1) AS avg_free_flow_sec,
    ROUND(AVG(t.current_travel_time - t.free_flow_travel_time)::numeric, 1) AS avg_extra_sec
FROM traffic_speeds_clean t
JOIN weather_clean w
  ON t.fetched_at::date = w.date
 AND EXTRACT(HOUR FROM t.fetched_at) = w.hour
GROUP BY w.is_monsoon;
"""
monsoon = sql(query)
monsoon["avg_extra_min"]          = monsoon["avg_extra_sec"].astype(float) / 60
# Assume 2 commutes/day × 5 days = 10 trips/week
monsoon["extra_hours_per_week"]   = (monsoon["avg_extra_min"] * 10 / 60).round(2)
monsoon


,is_monsoon,avg_travel_sec,avg_free_flow_sec,avg_extra_sec,avg_extra_min,extra_hours_per_week


---
##  Section 3 — Accidents & Safety


### Top Accident Hotspots

In [10]:
query = """
SELECT
    location,
    COUNT(*)       AS total_accidents,
    SUM(casualties) AS total_casualties
FROM accidents_clean
GROUP BY location
ORDER BY total_accidents DESC
LIMIT 15;
"""
hotspots = sql(query)
hotspots


,location,total_accidents,total_casualties
0,KR Puram,1,0.0
1,Electronic City Toll,1,2.0
2,Hebbal Flyover,1,2.0
3,Marathahalli Bridge,1,0.0
4,Outer Ring Road (Bellandur),1,3.0
5,Bannerghatta Road,1,0.0
6,Silk Board Junction,1,1.0
7,Koramangala 5th Block,1,1.0
8,MG Road,1,0.0


### Are the Most Congested Roads Also the Most Dangerous?

In [11]:
query = """
SELECT
    t.junction_label,
    ROUND(AVG(t.congestion_index)::numeric, 3) AS avg_congestion,
    COUNT(a.location)                                 AS accident_count
FROM traffic_speeds_clean t
LEFT JOIN accidents_clean a
  ON LOWER(a.location) LIKE '%' || LOWER(SPLIT_PART(t.junction_label, ' ', 1)) || '%'
GROUP BY t.junction_label
ORDER BY avg_congestion DESC;
"""
danger = sql(query)
danger


,junction_label,avg_congestion,accident_count
0,Outer Ring Road KR Puram,0.381,726
1,MG Road,0.380,726
2,Mahadevapura,0.379,0
3,Hebbal,0.378,726
4,Whitefield,0.378,0
5,Silk Board,0.378,726
6,Marathahalli,0.378,726
7,Koramangala,0.378,726
8,HSR Layout,0.377,0
9,BTM Layout,0.376,0


### Year-on-Year Accident Trend

In [12]:
query = """
SELECT
    year,
    COUNT(*) AS total_accidents,
    SUM(fatalities) AS total_fatalities,
    SUM(injuries)   AS total_injuries
FROM accidents_clean
WHERE year IS NOT NULL
GROUP BY year
ORDER BY year;
"""
yoy_accidents = sql(query)
yoy_accidents


,year,total_accidents,total_fatalities,total_injuries
0,2024,9,2.0,7.0


---
##  Section 4 — Transit Gaps & Infrastructure


### Transit Deserts — Wards with No BMTC Stop Within 500m

> 

In [13]:
# Build a rough grid of Bangalore wards and check BMTC stop density
stops_df = pd.read_sql("SELECT latitude, longitude FROM bmtc_stops_clean", engine)

# Create 1km x 1km grid cells and count stops per cell
stops_df["lat_bin"] = (stops_df["latitude"] // 0.009) * 0.009   # ~1km
stops_df["lon_bin"] = (stops_df["longitude"] // 0.009) * 0.009

stop_density = (
    stops_df.groupby(["lat_bin", "lon_bin"])
    .size()
    .reset_index(name="stop_count")
)

transit_deserts = stop_density[stop_density["stop_count"] == 0]
print(f"Grid cells with zero stops (transit deserts): {len(transit_deserts)}")

# Cells with very low coverage (1–2 stops)
sparse = stop_density[stop_density["stop_count"] <= 2]
print(f"Grid cells with ≤2 stops (sparse coverage):  {len(sparse)}")
stop_density.sort_values("stop_count").head(10)


Grid cells with zero stops (transit deserts): 0
Grid cells with ≤2 stops (sparse coverage):  9


,lat_bin,lon_bin,stop_count
0,12.834,77.670,1
1,12.897,77.598,1
2,12.915,77.616,1
3,12.933,77.616,1
4,12.933,77.670,1
5,12.951,77.697,1
6,12.969,77.607,1
7,12.996,77.688,1
8,13.041,77.589,1


### New Route Recommendations — High Congestion + Low Transit

In [14]:
query = """
SELECT
    junction_label,
    latitude,
    longitude,
    ROUND(AVG(congestion_index)::numeric, 3) AS avg_congestion
FROM traffic_speeds_clean
GROUP BY junction_label, latitude, longitude
ORDER BY avg_congestion DESC;
"""
traffic_df = sql(query)

# Simple spatial join: flag junctions that are far from nearest BMTC stop
from scipy.spatial import cKDTree
import numpy as np

stops = pd.read_sql("SELECT latitude, longitude FROM bmtc_stops_clean", engine)
tree = cKDTree(stops[["latitude", "longitude"]].values)

coords = traffic_df[["latitude", "longitude"]].values
dists, _ = tree.query(coords, k=1)
# 1° ≈ 111km → 0.0045° ≈ 500m
traffic_df["nearest_stop_deg"] = dists
traffic_df["transit_desert"]   = dists > 0.0045

recommendations = traffic_df[traffic_df["transit_desert"]].sort_values("avg_congestion", ascending=False)
print(f"Junctions with HIGH congestion AND poor transit coverage: {len(recommendations)}")
recommendations


Junctions with HIGH congestion AND poor transit coverage: 9


,junction_label,latitude,longitude,avg_congestion,nearest_stop_deg,transit_desert
0,Outer Ring Road KR Puram,12.9784,77.6408,0.381,0.031923,True
1,MG Road,12.9856,77.5533,0.380,0.056591,True
2,Mahadevapura,12.9902,77.7172,0.379,0.026309,True
3,Koramangala,12.9762,77.6033,0.378,0.005731,True
4,Whitefield,12.9698,77.7499,0.378,0.050910,True
6,Hebbal,13.0358,77.5970,0.378,0.009588,True
7,Silk Board,12.9716,77.5946,0.378,0.014945,True
8,HSR Layout,12.9719,77.5937,0.377,0.015741,True
9,BTM Layout,12.9141,77.6101,0.376,0.013077,True


### Top 5 Junctions Needing Signal Re-Timing

In [15]:
query = """
SELECT
    t.junction_label,
    ROUND(AVG(t.congestion_index)::numeric, 3)         AS avg_congestion,
    SUM(CASE WHEN t.congestion_index > 0.6 THEN 1 ELSE 0 END) AS high_congestion_hours,
    COUNT(a.location)                                         AS accident_count,
    ROUND(
        (AVG(t.congestion_index) * 0.6 +
         COUNT(a.location) * 0.4)::numeric, 3
    )                                                   AS priority_score
FROM traffic_speeds_clean t
LEFT JOIN accidents_clean a
  ON LOWER(a.location) LIKE '%' || LOWER(SPLIT_PART(t.junction_label, ' ', 1)) || '%'
GROUP BY t.junction_label
ORDER BY priority_score DESC
LIMIT 5;
"""
signal_retiming = sql(query)
print("Top 5 Junctions — Urgent Signal Re-Timing Needed:")
signal_retiming


Top 5 Junctions — Urgent Signal Re-Timing Needed:


,junction_label,avg_congestion,high_congestion_hours,accident_count,priority_score
0,Outer Ring Road KR Puram,0.381,145,726,290.629
1,MG Road,0.380,147,726,290.628
2,Hebbal,0.378,148,726,290.627
3,Silk Board,0.378,142,726,290.627
4,Marathahalli,0.378,142,726,290.627


### Road Infrastructure — Speed Bottlenecks


In [16]:
query = """
SELECT
    e.highway AS road_type,
    ROUND(AVG(e.maxspeed)::numeric, 1)      AS posted_limit_kmh,
    ROUND(AVG(t.current_speed)::numeric, 1) AS actual_speed_kmh,
    COUNT(*)                                AS data_points
FROM road_network_clean e
JOIN traffic_speeds_clean t
    ON LOWER(t.junction_label) LIKE '%' || LOWER(SPLIT_PART(e.name, ' ', 1)) || '%'
GROUP BY e.highway
HAVING COUNT(*) > 5
ORDER BY actual_speed_kmh ASC;
"""
speed_gap = sql(query)
speed_gap


,road_type,posted_limit_kmh,actual_speed_kmh,data_points
0,secondary,50.0,18.7,5082
1,tertiary,40.7,19.0,174240
2,residential,30.0,19.1,264264
3,"{unclassified,residential}",30.0,19.3,726
4,primary_link,56.0,20.3,3630
5,living_street,0.0,20.3,35574
6,trunk,60.7,21.0,148830
7,secondary_link,50.0,21.0,2178
8,unclassified,30.0,21.0,726
9,trunk_link,65.5,21.2,14520
